[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geo-di-lab/emerge-geoai-es/blob/main/docs/ch1/lesson5b.ipynb)

# Introducción a Google Earth Engine

**Para ejecutar el código:**

> Asegúrate de tener este cuaderno abierto en Google Colab. Si comienzas desde el libro digital, haz clic en [![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geo-di-lab/emerge-geoai-es/blob/main/docs/ch1/lesson5b.ipynb).

Cada bloque de código se denomina celda. Para ejecutar una celda, pasa el cursor sobre ella y haz clic en la flecha de la esquina superior izquierda, o haz clic dentro de la celda y presiona `Shift + Enter`.

Nota: La primera vez que ejecutes un bloque de código, Google Colab mostrará el mensaje `Warning: This notebook was not authored by Google`. Haz clic en `Run Anyway`.

Necesitas un proyecto de Google Earth Engine para ejecutar este código. Si todavía no lo has configurado, consulta la lección anterior, [Configurar Google Earth Engine](https://geo-di-lab.github.io/emerge-lessons/docs/ch1/lesson5a.html).

Una gran ventaja de Google Colab es que puedes escribir código en Python y ver el resultado directamente en el navegador. A continuación, repasaremos los conceptos básicos.

In [ ]:
import ee

# Autenticar Google Earth Engine
ee.Authenticate()

# Sustituye "emerge-lessons" por el ID de tu proyecto si es diferente
ee.Initialize(project="emerge-lessons")

En el código anterior, sustituye `emerge-lessons` por el ID de tu propio proyecto. Por ejemplo, si tu ID es `emerge-34956`, puedes cambiar la línea por la siguiente:

```python
ee.Initialize(project="emerge-34956")
```

### Terminología básica: uso de colecciones en GEE

Las **entidades** (*features*) son objetos geométricos acompañados de una lista de propiedades:

> `ee.Feature`

Las **imágenes** son similares a las entidades, pero pueden contener varias bandas:

> `ee.Image`

Las **colecciones** son grupos de entidades o imágenes:

> `ee.FeatureCollection` o `ee.ImageCollection`

En el [Catálogo de datos de Earth Engine](https://developers.google.com/earth-engine/datasets), la página de cada conjunto de datos incluye un fragmento de código que muestra cómo importarlo.

En este ejemplo utilizaremos el conjunto de datos [MODIS Terra Land Surface Temperature (LST)](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD11A1).

In [ ]:
# Importar la colección de temperatura de la superficie terrestre de MODIS
lst = ee.ImageCollection("MODIS/061/MOD11A1")

Los conjuntos de datos contienen distintos tipos de información distribuidos entre varias bandas. Algunos incluyen imágenes diarias con una resolución de 1 km, como el conjunto MODIS LST que utilizamos aquí, mientras que otros ofrecen una imagen anual con una resolución de 30 m.

La colección LST incluye, entre otras, las siguientes bandas:

- `LST_Day_1km`: temperatura diurna de la superficie terrestre.
- `Day_view_time`: hora local de la observación diurna.
- `LST_Night_1km`: temperatura nocturna de la superficie terrestre.

Como la colección contiene muchas imágenes, debemos aplicar filtros. Utiliza `filterDate()` para seleccionar imágenes dentro de un intervalo de fechas y `select()` para elegir variables específicas.

In [ ]:
# Fecha inicial de interés, incluida en el intervalo
i_date = "2024-01-01"

# Fecha final de interés, excluida del intervalo
f_date = "2024-01-31"

# Seleccionar las bandas y fechas apropiadas para LST
lst = (
    lst.select("LST_Day_1km", "QC_Day")
    .filterDate(i_date, f_date)
)

A continuación, definiremos dos puntos de interés para comparar. Como estamos estudiando la temperatura de la superficie terrestre, compararemos la temperatura de una zona urbana con la de una zona rural.

In [ ]:
# Definir un punto urbano cerca de Miami, Florida
urban_lon = -80.196432
urban_lat = 25.779766
urban_poi = ee.Geometry.Point(urban_lon, urban_lat)

# Definir un punto rural alejado de la ciudad, cerca de Homestead, Florida
rural_lon = -80.4998113
rural_lat = 25.3933527
rural_poi = ee.Geometry.Point(rural_lon, rural_lat)

Los valores del conjunto de datos MODIS que utilizamos deben multiplicarse por un factor de escala de 0.02 para obtener la temperatura en kelvin. Puedes consultar esta información en el [catálogo de Google Earth Engine](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD11A1#bands).

*Recuerda convertir los valores para obtener la respuesta final en las unidades deseadas.*

Necesitamos calcular la temperatura promedio de la superficie terrestre en los dos puntos de interés, por lo que utilizaremos `mean()`.

In [ ]:
# Escala en metros
scale = 1000


def get_mean_lst(poi, scale):
    """
    Calcula la temperatura promedio de la superficie terrestre
    para un punto y la convierte a grados Celsius.
    """
    mean_lst = (
        lst.mean()
        .sample(poi, scale)
        .first()
        .get("LST_Day_1km")
        .getInfo()
    )
    return round(mean_lst * 0.02 - 273.15, 2)


# Obtener la temperatura promedio para los puntos urbano y rural
urban_lst = get_mean_lst(urban_poi, scale)
rural_lst = get_mean_lst(rural_poi, scale)

print(
    "La temperatura promedio de la superficie terrestre "
    f"en el punto urbano es {urban_lst} °C."
)
print(
    "La temperatura promedio de la superficie terrestre "
    f"en el punto rural es {rural_lst} °C."
)

## Mapas interactivos

In [ ]:
import folium  # Crear mapas interactivos en Python
import geemap  # Otra opción para mapear datos de Google Earth Engine

Crea un mapa vacío centrado en Florida. Puedes hacer clic y arrastrar para desplazarte, y utilizar los controles para acercar o alejar la vista.

In [ ]:
map = folium.Map(
    location=[28.263363, -83.497652],
    tiles="Cartodb dark_matter",
    zoom_start=7
)

display(map)

A continuación, definiremos una función, adaptada de [este tutorial](https://developers.google.com/earth-engine/tutorials/community/intro-to-python-api), para agregar datos de Google Earth Engine a un mapa y mostrarlos de forma interactiva.

In [ ]:
def add_ee_layer(self, ee_image_object, vis_params, name):
    """
    Agrega a un mapa de Folium una capa de teselas
    procedente de una imagen de Earth Engine.
    """
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)

    folium.raster_layers.TileLayer(
        tiles=map_id_dict["tile_fetcher"].url_format,
        attr=(
            'Map Data &copy; '
            '<a href="https://earthengine.google.com/">'
            "Google Earth Engine</a>"
        ),
        name=name,
        overlay=True,
        control=True
    ).add_to(self)


folium.Map.add_ee_layer = add_ee_layer

In [ ]:
# Parámetros de visualización que definen los colores de los datos
lst_vis = {
    "min": 13000.0,
    "max": 16500.0,
    "palette": [
        "040274", "040281", "0502a3", "0502b8", "0502ce", "0502e6",
        "0602ff", "235cb1", "307ef3", "269db1", "30c8e2", "32d3ef",
        "3be285", "3ff38f", "86e26f", "3ae237", "b5e22e", "d6e21f",
        "fff705", "ffd611", "ffb613", "ff8b13", "ff6e08", "ff500d",
        "ff0000", "de0101", "c21301", "a71001", "911003"
    ]
}

# Agregar los datos al mapa
map.add_ee_layer(
    lst.mean().select("LST_Day_1km"),
    lst_vis,
    "Temperatura de la superficie terrestre"
)

display(map)

## Referencias

- [Creación de mapas interactivos con geemap](https://book.geemap.org/chapters/02_maps.html)
- [Introducción a la API de Python de Earth Engine](https://developers.google.com/earth-engine/tutorials/community/intro-to-python-api)